# Visão geral do pipeline climático

Este notebook percorre, passo a passo, a preparação de um ano de dados **fictícios** no formato do ERA5 antes de qualquer modelagem. Cada etapa chama uma função do pacote `pipeline_climatico`, a mesma usada por `python -m pipeline_climatico tudo`, e mostra o que ela verifica ou altera.

Execute a partir da raiz do repositório ou da pasta `notebooks/`. Nenhum resultado aqui é científico: os dados foram inventados para exercitar o fluxo.

In [ ]:
import os
import sys
from pathlib import Path

raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(raiz))
os.chdir(raiz)

import pandas as pd

from pipeline_climatico import ARQUIVO_ERA5, indice_era5, indice_horario_anual, inspecao, preparo

## 1. Inventário das variáveis

O arquivo usa os nomes curtos do Copernicus: `t2m`, `d2m` e `tp`. O CSV não diz a unidade nem se a grandeza é instantânea ou acumulada. Essa informação vem da documentação da fonte e precisa ser registrada antes de qualquer conta.

In [ ]:
inspecao.inventariar()

## 2. Validação da estrutura

`validar` interrompe o fluxo se faltar coluna ou se houver data vazia, repetida ou fora de ordem. Ela também conta os passos de 3 h que deveriam existir e não estão no arquivo, e confere uma regra física simples: o ponto de orvalho não passa da temperatura.

In [ ]:
pd.Series(inspecao.validar())

## 3. Passos ausentes e valores vazios

Um passo ausente é diferente de uma linha com valor vazio, e os dois precisam aparecer. Aqui eles simulam arquivos que não baixaram ou vieram corrompidos.

In [ ]:
bruto = pd.read_csv(ARQUIVO_ERA5, parse_dates=["valid_time"])
grade = indice_era5()
print(f"passos esperados: {len(grade)} (de 01/01 00h a 01/01/2025 00h)")
print(f"linhas no arquivo: {len(bruto)}")
print(f"passos sem registro: {len(grade.difference(bruto['valid_time']))}")
bruto[["t2m", "d2m", "tp"]].isna().sum().rename("valores vazios").to_frame()

## 4. Lacunas curtas e longas

A regra adotada interpola no tempo apenas lacunas internas de até 6 horas. Uma lacuna maior fica inteira como ausente, em vez de parcialmente preenchida. Na grade de 3 h, isso significa até 2 passos seguidos:

In [ ]:
indice = pd.date_range("2024-01-01", periods=9, freq="3h")
serie = pd.Series([20, None, 24, 25, None, None, None, None, 29], index=indice, dtype=float)
tratada, imputado = preparo.preencher_lacunas_curtas(serie, max_horas=6, passo_horas=3)
pd.DataFrame({"bruto": serie, "tratado": tratada, "imputado": imputado})

## 5. Unidades e umidade relativa

Temperatura e ponto de orvalho chegam em kelvin. A umidade relativa não existe no arquivo: ela é derivada de `t2m` e `d2m` pela fórmula de Magnus. Uma variável derivada herda as falhas das variáveis de origem.

In [ ]:
exemplo = bruto.dropna().head(3)
t = exemplo["t2m"] - 273.15
td = exemplo["d2m"] - 273.15
pd.DataFrame(
    {"t2m (K)": exemplo["t2m"], "temperatura (°C)": t, "orvalho (°C)": td, "UR (%)": preparo.umidade_relativa(t, td)}
).round(2)

## 6. De 3 h para 1 h: instantâneo e acumulado

Temperatura e orvalho são medidas de um instante e podem ser interpoladas entre dois passos vizinhos. A chuva `tp` é um **total** das 3 horas que terminam no horário indicado. Interpolar esse total para cada hora repete o valor quase três vezes e infla a chuva anual.

In [ ]:
tp_mm = bruto.set_index("valid_time")["tp"].reindex(grade) * 1000
horas = indice_horario_anual()

interpolando = tp_mm.reindex(horas.union(grade)).interpolate(method="time").reindex(horas)
repartindo = preparo.preparar_base()["precipitacao_mm"]

pd.Series(
    {
        "soma dos acumulados no ano (mm)": tp_mm.loc["2024-01-01 03:00":].sum() + tp_mm.iloc[0] / 3,
        "interpolando o acumulado (mm)": interpolando.sum(),
        "repartindo o acumulado em 3 h (mm)": repartindo.sum(),
    }
).round(1)

O acumulado das 00h de 1º de janeiro cobre 22h e 23h de 2023. Só um terço dele pertence ao ano. Pelo mesmo motivo, o arquivo traz o passo das 00h de 1º de janeiro de 2025: é ele que contém a chuva das 22h e 23h de 31 de dezembro.

A repartição usa o passo seguinte por no máximo 2 horas. Se um passo falta, as 3 horas que ele cobriria ficam ausentes; a chuva do vizinho nunca é emprestada.

## 7. Base processada

A base final tem uma linha por hora de 2024 e mantém todas as marcações. Quem usar a base depois consegue separar valor original, valor interpolado e valor ausente.

In [ ]:
base = preparo.preparar_base()
print(base.shape)
base.head()

In [ ]:
base.select_dtypes("number").describe().T.round(2)

In [ ]:
marcacoes = base.select_dtypes("bool").sum().rename("horas marcadas").to_frame()
marcacoes.loc["temperatura_c ausente"] = int(base["temperatura_c"].isna().sum())
marcacoes

Para gravar a base em `data/processed/` e gerar os painéis em `figures/`, execute na raiz:

```bash
python -m pipeline_climatico preparar
python -m pipeline_climatico graficos
```